# Adaptive Square-Dalitz integration for a narrow $\phi(1020)$ in $B^+\to K^-K^+K^+$

This notebook benchmarks the adaptive Square-Dalitz integrator on a deliberately simple amplitude model,

\[A = c_\phi A_\phi + c_{NR},\]

with a narrow $\phi(1020)\to K^-K^+$ and a constant nonresonant term. Because the two $K^+$ mesons are identical, the resonance amplitude is automatically symmetrized over the two physical $K^-K^+$ pairings.

The adaptive algorithm is **not** given the $\phi$ mass or width. It only inspects local convergence of the bilinears $J F_i^*F_j$.

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from dalitzplotfitter import (
    AdaptiveSquareDalitzGrid, DecayChannel, DecayModel, NonResonant,
    RealImag, Resonance, SquareDalitzGrid, enable_x64,
)

enable_x64()

## 1. Minimal $B^+\to K^-K^+K^+$ model

In [ ]:
channel = DecayChannel('B+', ('K-', 'K+', 'K+'))

PHI_MASS = 1.019461
PHI_WIDTH = 0.004249

model = DecayModel(
    channel,
    [
        Resonance(
            'phi1020', (0, 1), RealImag(1.0, 0.0),
            mass=PHI_MASS, width=PHI_WIDTH, spin=1,
            resonance_radius=1.5, parent_radius=5.0,
        ),
        NonResonant(RealImag(0.35, 0.20), name='NR'),
    ],
    normalize_components=False,
)

print('parent mass:', channel.parent_mass)
print('daughter masses:', channel.daughter_masses)
print('components:', [component.name for component in model.amplitude_model.components])

## 2. Helper: raw normalization matrix

The comparison uses the full complex matrix
\[M_{ij}=\int F_i^*F_j\,d\Phi,\]
not only the total normalization for one coefficient vector.

In [ ]:
def raw_matrix(sample):
    cache = model.prepare_cache(
        sample, normalization_sample=sample, normalize_components=False
    )
    return np.asarray(cache.normalization_matrix_fixed)

def relative_matrix_error(matrix, reference, floor=1e-8):
    scale = np.max(np.abs(reference))
    mask = np.abs(reference) > floor*scale
    relative = np.zeros_like(np.abs(reference), dtype=float)
    relative[mask] = np.abs(matrix[mask] - reference[mask]) / np.abs(reference[mask])
    return relative, float(np.max(relative[mask]))

def print_matrix(label, matrix):
    print(label)
    for row in matrix:
        print('  ', '  '.join(f'{z.real:+.6e}{z.imag:+.6e}j' for z in row))

## 3. Dense uniform midpoint reference

A dense uniform SqDP grid is used only as a numerical reference for this benchmark.

In [ ]:
PAIR = (0, 1)
REFERENCE_N = 700

reference_sample = SquareDalitzGrid(
    channel.parent_mass, channel.daughter_masses,
    resolution=REFERENCE_N, pair=PAIR, quadrature='midpoint',
).sample()
M_reference = raw_matrix(reference_sample)

print('reference points:', reference_sample.size)
print_matrix('M reference', M_reference)

## 4. Convergence of uniform midpoint grids

In [ ]:
uniform_resolutions = [60, 80, 120, 180, 250, 350, 500]
uniform_rows = []

for n in uniform_resolutions:
    sample = SquareDalitzGrid(
        channel.parent_mass, channel.daughter_masses,
        resolution=n, pair=PAIR, quadrature='midpoint',
    ).sample()
    matrix = raw_matrix(sample)
    _, max_error = relative_matrix_error(matrix, M_reference)
    uniform_rows.append((n, sample.size, max_error, matrix))
    print(f'N={n:4d}  points={sample.size:8d}  max matrix rel. error={max_error:.6e}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog([row[1] for row in uniform_rows], [row[2] for row in uniform_rows], marker='o')
ax.set(xlabel='normalization points', ylabel='max relative matrix-element error', title='Uniform midpoint convergence')
ax.grid(True, which='both', alpha=0.3)
plt.show()

## 5. Build the adaptive grid

The base grid is deliberately coarse. `min_depth=1` guarantees one global subdivision before the error criterion is allowed to stop refinement, reducing the chance that an ultra-narrow feature lies entirely between the first cell centers.

In [ ]:
adaptive_builder = AdaptiveSquareDalitzGrid(
    channel.parent_mass, channel.daughter_masses,
    pair=PAIR,
    base_resolution=18,
    min_depth=1,
    max_depth=6,
    tolerance=0.02,
    matrix_floor=1e-9,
    max_cells=200_000,
)

adaptive = adaptive_builder.build(model)
M_adaptive = raw_matrix(adaptive.sample)
adaptive_element_errors, adaptive_max_error = relative_matrix_error(M_adaptive, M_reference)

print('adaptive leaves:', adaptive.n_leaves)
print('adaptive integration points:', adaptive.size)
print('max leaf depth:', int(adaptive.leaf_depths.max()))
print('max matrix relative error:', adaptive_max_error)
print_matrix('M adaptive', M_adaptive)

## 6. Where did the algorithm refine?

The mesh should concentrate near the narrow $K^-K^+$ structures generated by the symmetrized $\phi(1020)$ amplitude, although no pole metadata was supplied to the integrator.

In [ ]:
bounds = adaptive.leaf_bounds
segments = []
for x0, x1, y0, y1 in bounds:
    segments.extend([
        [(x0, y0), (x1, y0)], [(x1, y0), (x1, y1)],
        [(x1, y1), (x0, y1)], [(x0, y1), (x0, y0)],
    ])

fig, ax = plt.subplots(figsize=(7, 7))
ax.add_collection(LineCollection(segments, linewidths=0.25))
ax.set(xlim=(0,1), ylim=(0,1), xlabel=r'$m^\prime$', ylabel=r'$\theta^\prime$', title='Adaptive SqDP leaf cells')
ax.set_aspect('equal')
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
centers_x = 0.5*(bounds[:,0] + bounds[:,1])
centers_y = 0.5*(bounds[:,2] + bounds[:,3])
scatter = ax.scatter(centers_x, centers_y, c=adaptive.leaf_depths, s=6)
fig.colorbar(scatter, ax=ax, label='leaf depth')
ax.set(xlim=(0,1), ylim=(0,1), xlabel=r'$m^\prime$', ylabel=r'$\theta^\prime$', title='Adaptive refinement depth')
plt.show()

## 7. Efficiency versus a uniform grid

In [ ]:
comparison = []
for n, npoints, error, _ in uniform_rows:
    comparison.append(('uniform', npoints, error, n))
comparison.append(('adaptive', adaptive.size, adaptive_max_error, None))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog([row[1] for row in uniform_rows], [row[2] for row in uniform_rows], 'o-', label='uniform midpoint')
ax.scatter([adaptive.size], [adaptive_max_error], marker='*', s=180, label='adaptive')
ax.set(xlabel='normalization points', ylabel='max relative matrix-element error', title=r'$\phi(1020)+NR$: accuracy versus cost')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.show()

print(f'adaptive: {adaptive.size} points, max error={adaptive_max_error:.3e}')
closest = min(uniform_rows, key=lambda row: abs(row[1]-adaptive.size))
print(f'closest uniform: N={closest[0]}, {closest[1]} points, max error={closest[2]:.3e}')

## 8. Inspect the individual matrix elements

In [ ]:
names = ['phi1020', 'NR']
for i, name_i in enumerate(names):
    for j, name_j in enumerate(names):
        ref = M_reference[i,j]
        val = M_adaptive[i,j]
        denom = max(abs(ref), 1e-15)
        print(f'{name_i:8s} x {name_j:8s}: ref={ref:+.8e}, adaptive={val:+.8e}, rel.err={abs(val-ref)/denom:.3e}')

## Interpretation

The important quantity is the complete normalization matrix, including the $\phi$--NR interference. A useful adaptive configuration should reach an error comparable to a much denser uniform grid with substantially fewer points.

The refinement is driven solely by numerical behavior of $F_i^*F_j$. Therefore the same algorithm can later be used for non-Breit-Wigner components that do not expose a pole mass or width. For production fits the tolerance and maximum depth should be validated by repeating this matrix-element convergence study with the complete physics model.